# Data Exploration and Visualization

This notebook explores the nurse scheduling datasets, visualizing demand patterns, analyzing shift distributions, and examining scenario variability.

**Dataset:** Paper validation data (20 nurses, 50 scenarios, 28 days)

**Objectives:**
1. Load and validate input data quality
2. Analyze demand patterns by shift type and day
3. Examine weekend vs weekday differences
4. Visualize scenario variability
5. Understand nurse availability and skills

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Load datasets
nurses_df = pd.read_csv('data/paper_validation_nurses.csv')
scenarios_df = pd.read_csv('data/paper_validation_scenarios.csv')

print(f"✅ Loaded {len(nurses_df)} nurses")
print(f"✅ Loaded {len(scenarios_df)} scenario records ({scenarios_df['scenario'].nunique()} scenarios)")
print(f"✅ Planning horizon: {scenarios_df['day'].nunique()} days")

## 2. Nurse Data Overview

In [ ]:
# Display nurse information
print("Nurse Dataset Preview:")
print(nurses_df.head(10))
print("\n" + "="*80 + "\n")

# Statistical summary
print("Nurse Skills Distribution:")
skill_counts = nurses_df['skill'].value_counts().sort_index()
for skill, count in skill_counts.items():
    print(f"  Skill Level {skill}: {count} nurses ({count/len(nurses_df)*100:.1f}%)")

# Visualize skill distribution
fig = px.pie(nurses_df, names='skill', title='Nurse Skills Distribution',
             labels={'skill': 'Skill Level'},
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

## 3. Scenario Data Analysis

In [ ]:
# Display scenario data structure
print("Scenario Dataset Preview:")
print(scenarios_df.head(10))
print("\n" + "="*80 + "\n")

# Compute aggregate statistics
print("Demand Statistics:")
print(f"  Total scenarios: {scenarios_df['scenario'].nunique()}")
print(f"  Days per scenario: {scenarios_df['day'].nunique()}")
print(f"  Shift types: {[col for col in scenarios_df.columns if col.startswith('demand_')]}")
print(f"\n  Average daily demand:")
for shift in ['morning', 'afternoon', 'night', 'extra']:
    avg_demand = scenarios_df[f'demand_{shift}'].mean()
    print(f"    {shift.capitalize()}: {avg_demand:.2f} nurses")

# Total demand across all shifts
scenarios_df['total_demand'] = (scenarios_df['demand_morning'] + 
                                 scenarios_df['demand_afternoon'] + 
                                 scenarios_df['demand_night'] + 
                                 scenarios_df['demand_extra'])
print(f"\n  Average total daily demand: {scenarios_df['total_demand'].mean():.2f} nurses")

## 4. Demand Patterns Over Time

In [ ]:
# Calculate average demand per day across all scenarios
avg_demand_by_day = scenarios_df.groupby('day')[['demand_morning', 'demand_afternoon', 
                                                   'demand_night', 'demand_extra']].mean()

# Plot demand over time
fig = go.Figure()
shifts = ['morning', 'afternoon', 'night', 'extra']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for shift, color in zip(shifts, colors):
    fig.add_trace(go.Scatter(
        x=avg_demand_by_day.index,
        y=avg_demand_by_day[f'demand_{shift}'],
        name=shift.capitalize(),
        line=dict(color=color, width=2),
        mode='lines+markers'
    ))

fig.update_layout(
    title='Average Demand by Shift Type Over 28 Days',
    xaxis_title='Day',
    yaxis_title='Average Demand (nurses)',
    hovermode='x unified',
    height=500
)
fig.show()

## 5. Weekend vs Weekday Analysis

In [ ]:
# Classify days as weekend (Saturday=6, Sunday=7) or weekday
scenarios_df['is_weekend'] = scenarios_df['day'].apply(lambda d: 'Weekend' if d % 7 in [6, 0] else 'Weekday')

# Compare average demand
weekend_comparison = scenarios_df.groupby('is_weekend')[['demand_morning', 'demand_afternoon', 
                                                          'demand_night', 'demand_extra', 
                                                          'total_demand']].mean()

print("Weekend vs Weekday Demand Comparison:")
print(weekend_comparison)
print("\n" + "="*80 + "\n")

# Visualize comparison
fig = go.Figure()
shifts = ['morning', 'afternoon', 'night', 'extra']
weekday_data = [weekend_comparison.loc['Weekday', f'demand_{s}'] for s in shifts]
weekend_data = [weekend_comparison.loc['Weekend', f'demand_{s}'] for s in shifts]

fig.add_trace(go.Bar(name='Weekday', x=[s.capitalize() for s in shifts], y=weekday_data))
fig.add_trace(go.Bar(name='Weekend', x=[s.capitalize() for s in shifts], y=weekend_data))

fig.update_layout(
    title='Average Demand: Weekday vs Weekend',
    xaxis_title='Shift Type',
    yaxis_title='Average Demand (nurses)',
    barmode='group',
    height=400
)
fig.show()

## 6. Scenario Variability Analysis

In [ ]:
# Analyze demand variability across scenarios for each day
variability_data = []

for day in range(1, 29):
    day_data = scenarios_df[scenarios_df['day'] == day]
    for shift in ['morning', 'afternoon', 'night', 'extra']:
        variability_data.append({
            'day': day,
            'shift': shift,
            'mean': day_data[f'demand_{shift}'].mean(),
            'std': day_data[f'demand_{shift}'].std(),
            'min': day_data[f'demand_{shift}'].min(),
            'max': day_data[f'demand_{shift}'].max()
        })

variability_df = pd.DataFrame(variability_data)

# Plot coefficient of variation (CV = std/mean)
variability_df['cv'] = variability_df['std'] / variability_df['mean']

fig = px.box(variability_df, x='shift', y='cv', 
             title='Demand Variability by Shift Type (Coefficient of Variation)',
             labels={'cv': 'Coefficient of Variation', 'shift': 'Shift Type'})
fig.show()

print("\nVariability Summary (Coefficient of Variation):")
print(variability_df.groupby('shift')['cv'].describe())

## 7. Heatmap: Demand Patterns

In [ ]:
# Create pivot table for heatmap (average demand per day and shift)
heatmap_data = scenarios_df.groupby('day')[['demand_morning', 'demand_afternoon', 
                                             'demand_night', 'demand_extra']].mean()
heatmap_data.columns = ['Morning', 'Afternoon', 'Night', 'Extra']

# Create heatmap
fig = px.imshow(heatmap_data.T, 
                labels=dict(x="Day", y="Shift Type", color="Demand"),
                x=heatmap_data.index,
                y=heatmap_data.columns,
                color_continuous_scale='YlOrRd',
                aspect='auto')
fig.update_layout(title='Average Demand Heatmap (28 Days × 4 Shifts)', height=400)
fig.show()

## 8. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values Check:")
print(f"  Nurses dataset: {nurses_df.isnull().sum().sum()} missing values")
print(f"  Scenarios dataset: {scenarios_df.isnull().sum().sum()} missing values")

# Check for negative demands (should be none)
print(f"\nNegative Demand Check:")
for shift in ['morning', 'afternoon', 'night', 'extra']:
    negative_count = (scenarios_df[f'demand_{shift}'] < 0).sum()
    print(f"  {shift.capitalize()}: {negative_count} negative values")

# Check data ranges
print(f"\nData Range Validation:")
print(f"  Nurse IDs: {nurses_df['nurse_id'].min()} to {nurses_df['nurse_id'].max()}")
print(f"  Skill levels: {nurses_df['skill'].min()} to {nurses_df['skill'].max()}")
print(f"  Scenario IDs: {scenarios_df['scenario'].min()} to {scenarios_df['scenario'].max()}")
print(f"  Days: {scenarios_df['day'].min()} to {scenarios_df['day'].max()}")

# Verify scenario completeness (each scenario should have 28 days)
scenario_day_counts = scenarios_df.groupby('scenario')['day'].count()
incomplete_scenarios = scenario_day_counts[scenario_day_counts != 28]
if len(incomplete_scenarios) > 0:
    print(f"\n⚠️  WARNING: {len(incomplete_scenarios)} scenarios have incomplete data")
else:
    print(f"\n✅ All {len(scenario_day_counts)} scenarios have complete 28-day data")

print("\n" + "="*80)
print("Data quality check complete!")